# 04.05 Document Classification and Linkage

**Phase 04 — Document Processing & OCR**  
**Objective:** Classify supporting documents into canonical OFPHIL categories using OCR text + keyword/TF-IDF confidence scoring, link documents to patient records without target leakage, detect duplicate/conflicting or missing required documents, and export structured outputs for Notebook 06 and Phase 05.


### 1. Environment Setup & Input Contract Verification


In [ ]:
import os
import sys
import json
import joblib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

def _find_root(p: Path = Path.cwd()) -> Path:
    for parent in [p.resolve(), *p.resolve().parents]:
        if (parent / 'data').exists() and (parent / 'experiments').exists():
            return parent
    return p.resolve()

PROJECT_ROOT = _find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.document_processing import DocumentProcessingPaths
paths = DocumentProcessingPaths(PROJECT_ROOT)

OCR_ENTITIES_JSON = paths.interim_doc_proc / 'extracted_ocr' / 'ocr_extracted_entities.json'
LINKAGE_MANIFEST = paths.patient_linkage_manifest
MODEL_OUT_DIR = paths.doc_clf_model_dir
FEATURES_OUT_DIR = paths.processed_dir / 'features'
CLASSIFICATION_JSON = paths.interim_doc_proc / 'document_classification_linkage.json'
DERIVED_FEATURES_CSV = FEATURES_OUT_DIR / 'document_derived_features.csv'

MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURES_OUT_DIR.mkdir(parents=True, exist_ok=True)

CANONICAL_CATEGORIES = [
    'certificate_of_indigency',
    'medical_certificate',
    'government_id',
    'electricity_bill',
    'water_bill',
    'referral_form',
    'expense_quotation',
    'case_study',
]
REQUIRED_CHARITY_DOCS = {'certificate_of_indigency', 'medical_certificate', 'government_id'}

print(f"Project Root        : {paths.root}")
print(f"OCR Entities JSON   : {OCR_ENTITIES_JSON} (exists={OCR_ENTITIES_JSON.exists()})")
print(f"Linkage Manifest    : {LINKAGE_MANIFEST} (exists={LINKAGE_MANIFEST.exists()})")
print(f"Classifier Artifacts: {MODEL_OUT_DIR}")


### 2. Document Category Classification & Confidence Scoring


In [ ]:
CATEGORY_KEYWORDS = {
    'certificate_of_indigency': ['indigency', 'indigent', 'barangay', 'punong barangay', 'katibayan', 'residency'],
    'medical_certificate': ['medical certificate', 'diagnosis', 'physician', 'hospital', 'ophthalmology', 'vitrectomy', 'cataract', 'clinic'],
    'government_id': ['republic of the philippines', 'identification', 'senior citizen', 'pwd', 'sss', 'philhealth', 'driver', 'passport'],
    'electricity_bill': ['meralco', 'electric', 'kwh', 'kilowatt', 'power', 'electricity bill'],
    'water_bill': ['maynilad', 'manila water', 'water district', 'cubic meter', 'water bill'],
    'referral_form': ['referral', 'financial assistance', 'social service', 'msw', 'endorsement'],
    'expense_quotation': ['quotation', 'estimate', 'surgical fee', 'iol', 'anesthesia', 'operating room'],
    'case_study': ['case study', 'social case', 'family composition', 'monthly income', 'guardian'],
}

def classify_document_text(text: str, folder_hint: str = '') -> tuple:
    t = str(text or '').lower()
    scores = {}
    for cat, kws in CATEGORY_KEYWORDS.items():
        hits = sum(1 for kw in kws if kw in t)
        scores[cat] = hits / float(max(1, len(kws)))
    best_cat = max(scores, key=scores.get)
    best_score = scores[best_cat]
    if best_score > 0:
        conf = round(min(0.98, 0.55 + best_score * 0.65), 4)
        return best_cat, conf, 'OCR_KEYWORD_SCORING'
    if folder_hint and folder_hint in CANONICAL_CATEGORIES:
        return folder_hint, 0.60, 'FOLDER_TAXONOMY_FALLBACK'
    return 'unclassified', 0.25, 'NEEDS_MANUAL_REVIEW'

# Load OCR extracted entities (from Notebook 04) & linkage manifest
ocr_entities = json.loads(OCR_ENTITIES_JSON.read_text(encoding='utf-8')) if OCR_ENTITIES_JSON.exists() else {}
df_links = pd.read_csv(LINKAGE_MANIFEST) if LINKAGE_MANIFEST.exists() else pd.DataFrame()

classified_rows = []
for doc_id, rec in ocr_entities.items():
    pred_cat, conf, method = classify_document_text(rec.get('full_text_preview', ''), '')
    classified_rows.append({
        'document_id': doc_id,
        'file_name': rec.get('file_name', doc_id),
        'predicted_category': pred_cat,
        'classification_confidence': conf,
        'classification_method': method,
        'ocr_avg_confidence': rec.get('average_confidence', 0.0),
        'needs_manual_review': bool(conf < 0.70 or rec.get('average_confidence', 0.0) < 0.75),
    })

if not classified_rows and not df_links.empty:
    for _, row in df_links.iterrows():
        hint = str(row.get('document_category', row.get('document_type', ''))).lower()
        pred_cat, conf, method = classify_document_text(str(row.get('filename', '')), hint)
        classified_rows.append({
            'document_id': str(row.get('document_id', '')),
            'file_name': str(row.get('filename', '')),
            'predicted_category': pred_cat,
            'classification_confidence': conf,
            'classification_method': method,
            'ocr_avg_confidence': np.nan,
            'needs_manual_review': bool(conf < 0.70),
        })

df_classified = pd.DataFrame(classified_rows)
print(f"Classified {len(df_classified):,} documents.")
display(df_classified.head(10))


### 3. Patient-Document Linkage, Conflict Detection & Leakage-Free Feature Export


In [ ]:
linkage_audit = {
    'total_documents_classified': int(len(df_classified)),
    'category_distribution': df_classified['predicted_category'].value_counts().to_dict() if not df_classified.empty else {},
    'duplicate_document_ids': int(df_classified['document_id'].duplicated().sum()) if not df_classified.empty else 0,
    'needs_manual_review_count': int(df_classified['needs_manual_review'].sum()) if not df_classified.empty else 0,
    'linkage_records_present': int(len(df_links)),
}

# Build leakage-free patient-level document presence features (strictly pre-decision indicators; no eligibility labels)
patient_col = next((c for c in ['patient_id', 'record_id', 'case_id'] if c in df_links.columns), None)
doc_type_col = next((c for c in ['document_category', 'document_type'] if c in df_links.columns), None)

patient_features = []
if patient_col and doc_type_col and not df_links.empty:
    for pid, grp in df_links.groupby(patient_col):
        dtypes = set(grp[doc_type_col].dropna().astype(str).str.lower().tolist())
        missing_req = sorted(list(REQUIRED_CHARITY_DOCS - dtypes))
        patient_features.append({
            'patient_id': pid,
            'linked_document_count': int(len(grp)),
            'has_indigency_document': int('certificate_of_indigency' in dtypes),
            'has_medical_certificate': int('medical_certificate' in dtypes or 'medical_abstract' in dtypes),
            'has_valid_id': int('government_id' in dtypes),
            'has_utility_bill': int('electricity_bill' in dtypes or 'water_bill' in dtypes),
            'missing_required_doc_count': int(len(missing_req)),
            'missing_required_docs': ', '.join(missing_req),
            'duplicate_category_flag': int(grp[doc_type_col].duplicated().any()),
        })

df_doc_features = pd.DataFrame(patient_features)
if not df_doc_features.empty:
    df_doc_features.to_csv(DERIVED_FEATURES_CSV, index=False, encoding='utf-8')
    print(f"[OK] Exported leakage-free document features ({len(df_doc_features):,} patients) -> {DERIVED_FEATURES_CSV}")
    display(df_doc_features.head(10))

out_payload = {
    'audit_summary': linkage_audit,
    'documents': df_classified.to_dict('records'),
}
with open(CLASSIFICATION_JSON, 'w', encoding='utf-8') as f:
    json.dump(out_payload, f, indent=2)
with open(MODEL_OUT_DIR / 'taxonomy_schema.json', 'w', encoding='utf-8') as f:
    json.dump({'canonical_categories': CANONICAL_CATEGORIES, 'required_charity_docs': sorted(REQUIRED_CHARITY_DOCS)}, f, indent=2)

print(f"[OK] Saved classification & linkage contract -> {CLASSIFICATION_JSON}")
